# Hyperparameter Tuning Examples

This notebook demonstrates how to perform hyperparameter optimization for POMDP planners using the POMDPPlanners framework. We'll show how to optimize different algorithms on various environments using Optuna-based optimization.

## Overview

Hyperparameter tuning is crucial for achieving optimal performance in POMDP planning. The framework provides a comprehensive hyperparameter optimization system that:

- Uses Optuna for efficient parameter search
- Supports both numerical and categorical parameters
- Provides MLflow integration for experiment tracking
- Handles multiple environments and algorithms simultaneously
- Includes statistical analysis and confidence intervals

## Basic Hyperparameter Optimization

Let's start with a simple example optimizing POMCP on the Tiger POMDP:

In [ ]:
from pathlib import Path
from POMDPPlanners.simulations.simulation_apis.local_simulations_api import LocalSimulationsAPI
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs
from POMDPPlanners.core.simulation import (
    NumericalHyperParameter, CategoricalHyperParameter
)
from POMDPPlanners.core.simulation.hyperparameter_tuning import (
    HyperParamPlannerConfig, HyperParameterRunParams, HyperParameterOptimizationDirection,
)
from POMDPPlanners.planners.mcts_planners.pomcpow import POMCPOW
from POMDPPlanners.utils.action_samplers import DiscreteActionSampler
import random

# Initialize the API
api = LocalSimulationsAPI(
    cache_dir_path=Path("./hyperparameter_results"),
    debug=True
)

# Create environment configuration
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
tiger_env, tiger_belief = env_configs.tiger_pomdp_config(n_particles=50)  # Minimal for testing

action_sampler = DiscreteActionSampler(tiger_env.get_actions())

print(f"Created Tiger POMDP environment: {tiger_env.__class__.__name__}")
print(f"Initial belief has {len(tiger_belief.particles)} particles")

In [ ]:
# Define hyperparameter optimization configuration
optimization_configs = [
    HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        hyper_param_planner_config=HyperParamPlannerConfig(
            policy_cls=POMCPOW,
            hyper_parameters=[
                NumericalHyperParameter(0.1, 100., "exploration_constant"),  # Reduced range
                NumericalHyperParameter(2, 4, "depth"),  # Reduced depth
                NumericalHyperParameter(1.0, 5.0, "k_o"),  # Observation progressive widening coefficient
                NumericalHyperParameter(1.0, 5.0, "k_a"),  # Action progressive widening coefficient
                NumericalHyperParameter(0.01, 0.5, "alpha_o"),  # Observation progressive widening exponent
                NumericalHyperParameter(0.01, 0.5, "alpha_a")   # Action progressive widening exponent
            ],
            constant_parameters={
                "discount_factor": 0.95,
                "n_simulations": 2000,  # Minimal simulations for testing
                "action_sampler": action_sampler,
                "name": "OptimizedPOMCPOW_Tiger"
            },
        ),
        num_episodes=10,       # Minimal episodes for testing
        num_steps=10,          # Minimal steps for testing
        n_trials=200,          # Minimal trials for testing
        parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
    )
]

print("Hyperparameter optimization configuration:")
for i, config in enumerate(optimization_configs):
    print(f"  Config {i+1}: {config.hyper_param_planner_config.policy_cls.__name__} with {len(config.hyper_param_planner_config.hyper_parameters)} parameters")
    print(f"    Trials: {config.n_trials}, Episodes: {config.num_episodes}")

In [ ]:
# Run hyperparameter optimization
print("Starting hyperparameter optimization...")
results = api.run_hyperparameter_optimization(
    environment_run_params=optimization_configs,
    experiment_name="Tiger_POMCPOW_Optimization",
    n_jobs=1,  # Use single core for testing
)

# Analyze results
print("\n=== Optimization Results ===")
for i, result in enumerate(results):
    print(f"Configuration {i+1} Results:")
    print(f"  Environment: {result.environment.__class__.__name__}")
    print(f"  Policy: {result.policy.__class__.__name__}")
    print(f"  Best hyperparameters: {result.chosen_hyper_parameters}")
    print(f"  Policy name: {result.policy.name}")
    print()

## Multi-Environment Optimization

Now let's optimize multiple algorithms on different environments:

In [ ]:
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
import numpy as np

# Initialize APIs
api = LocalSimulationsAPI(
    cache_dir_path=Path("./multi_env_optimization"),
    debug=True
)

env_configs = EnvironmentConfigsAPI(discount_factor=0.95)

# Create environments (using Tiger environment for both to ensure compatibility)
rock_sample_env, rock_sample_belief = tiger_env, tiger_belief
laser_tag_env, laser_tag_belief = tiger_env, tiger_belief

print(f"Using Tiger environment for both configurations")
print(f"Environment: {rock_sample_env.__class__.__name__}")
print(f"Belief particles: {len(rock_sample_belief.particles)}")

In [4]:
# Define multiple optimization configurations
multi_env_optimization_configs = []

# SparsePFT configuration
sparse_pft_config = HyperParameterRunParams(
    environment=rock_sample_env,
    belief=rock_sample_belief,
    hyper_param_planner_config=HyperParamPlannerConfig(
        policy_cls=SparsePFT,
        hyper_parameters=[
            NumericalHyperParameter(2, 4, "depth"),  # Minimal depth
            NumericalHyperParameter(0.0, 10.0, "c_ucb"),  # Reduced range
            NumericalHyperParameter(0.0, 10.0, "beta_ucb")  # Reduced range
        ],
        constant_parameters={
            "discount_factor": 0.95,
            "gamma": 0.95,
            "belief_child_num": 1,  # Minimal
            "n_simulations": 50,  # Minimal simulations
            "name": "OptimizedSparsePFT"
        },
    ),
    num_episodes=10,  # Minimal episodes
    num_steps=10,     # Minimal steps
    n_trials=200,      # Minimal trials
    parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
)
multi_env_optimization_configs.append(sparse_pft_config)
print("Added SparsePFT configuration")

# POMCPOW configuration for comparison
pomcpow_config = HyperParameterRunParams(
    environment=laser_tag_env,
    belief=laser_tag_belief,
    hyper_param_planner_config=HyperParamPlannerConfig(
        policy_cls=POMCPOW,
        hyper_parameters=[
            NumericalHyperParameter(0.1, 100.0, "exploration_constant"),  # Reduced range
            NumericalHyperParameter(2, 15, "depth"),  # Minimal depth
            NumericalHyperParameter(1.0, 4.0, "k_o"),  # Observation progressive widening coefficient
            NumericalHyperParameter(1.0, 4.0, "k_a"),  # Action progressive widening coefficient
            NumericalHyperParameter(0.01, 0.5, "alpha_o"),  # Observation progressive widening exponent
            NumericalHyperParameter(0.01, 0.5, "alpha_a")   # Action progressive widening exponent
        ],
        constant_parameters={
            "discount_factor": 0.95,
            "n_simulations": 2000,  # Minimal simulations
            "action_sampler": action_sampler,
            "name": "OptimizedPOMCPOW_Multi"
        },
    ),
    num_episodes=10,  # Minimal episodes
    num_steps=10,     # Minimal steps
    n_trials=200,      # Minimal trials
    parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
)
multi_env_optimization_configs.append(pomcpow_config)
print("Added POMCPOW configuration")

print(f"\nTotal configurations: {len(multi_env_optimization_configs)}")

Added SparsePFT configuration
Added POMCPOW configuration

Total configurations: 2


In [ ]:
# Run multi-environment optimization
print("Starting multi-environment optimization...")
multi_results = api.run_hyperparameter_optimization(
    environment_run_params=multi_env_optimization_configs,
    experiment_name="Multi_Environment_Algorithm_Optimization",
    n_jobs=-1,  # Use single core for testing
)

# Analyze and compare results
print("\n=== Multi-Environment Optimization Results ===")
for i, result in enumerate(multi_results):
    env_name = result.environment.__class__.__name__
    policy_name = result.policy.__class__.__name__
    best_params = result.chosen_hyper_parameters
    
    print(f"\nConfiguration {i+1}: {policy_name} on {env_name}")
    print(f"  Best hyperparameters: {best_params}")
    print(f"  Policy name: {result.policy.name}")

2025-10-06 16:13:24,477 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:805 - Starting hyperparameter optimization for 2 configurations
2025-10-06 16:13:24,478 - DEBUG: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:808 - Parameters: experiment_name=Multi_Environment_Algorithm_Optimization, n_jobs=-1, confidence_interval=0.95, alpha=0.05
2025-10-06 16:13:24,513 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:835 - Running hyperparameter optimization
2025-10-06 16:13:24,514 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/hyper_parameter_tuning_simulations.py:403 - Starting optimization for 2 configurations using stub interface with MLflow tracking
2025-10-06 16:13:24,521 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/simulation/tasks.py:306 - Starting to process 2 tasks
2025-10-06 16:13:24,522 - INFO: /hom

Starting multi-environment optimization...


Running tasks:   0%|          | 0/2 [00:00<?, ?it/s]2025-10-06 16:13:24,527 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:162 - Starting hyperparameter optimization task with Pareto selection
2025-10-06 16:13:24,527 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:165 - Environment: TigerPOMDP (config_id: aa8cb97f2fbfd131a356760c577a7c5a88efc7abc9c4a585062113bfe5da7d45)
2025-10-06 16:13:24,527 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:168 - Policy class: SparsePFT
2025-10-06 16:13:24,528 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:177 - Parameters to optimize: average_return (maximize)


## Using Predefined Hyperparameter Configurations

The framework provides predefined hyperparameter configurations for common algorithms:

In [ ]:
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs

# Initialize configuration APIs
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
planner_configs = PlannersHyperparamConfigs(discount_factor=0.95)

# Use predefined configurations - note: using POMCP config as POMCPOW may not have predefined configs
pomcp_predefined = planner_configs.pomcp_config(
    env=tiger_env, 
    name="PredefinedPOMCP_Tiger"
)

print("Predefined POMCP configuration:")
print(f"  Policy class: {pomcp_predefined.policy_cls.__name__}")
print(f"  Hyperparameters: {len(pomcp_predefined.hyper_parameters)} parameters")
for hp in pomcp_predefined.hyper_parameters:
    print(f"    - {hp.name}: {getattr(hp, 'low', 'categorical')} to {getattr(hp, 'high', getattr(hp, 'choices', 'N/A'))}")
print(f"  Constant parameters: {list(pomcp_predefined.constant_parameters.keys())}")
    

# Convert to optimization parameters if available
predefined_optimization_config = HyperParameterRunParams(
    environment=tiger_env,
    belief=tiger_belief,
    hyper_param_planner_config=pomcp_predefined,
    num_episodes=10,  # Increased from 5 for better statistics
    num_steps=10,
    n_trials=200,  # Reduced for testing
    parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
)
    
print("\nRunning optimization with predefined configuration...")
predefined_results = api.run_hyperparameter_optimization(
    environment_run_params=[predefined_optimization_config],
    experiment_name="Predefined_Config_Optimization",
    n_jobs=-1,
)

print("\n=== Predefined Configuration Results ===")
for result in predefined_results:
    print(f"Policy: {result.policy.__class__.__name__}")
    print(f"Best parameters: {result.chosen_hyper_parameters}")

## Analyzing Optimization Results

After optimization, you can analyze the results and use the optimized policies:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract optimization results
all_results = results  # Use the first optimization results
optimized_policies = [result.policy for result in all_results]

# Create a comparison table
comparison_data = []
for i, result in enumerate(all_results):
    comparison_data.append({
        'Configuration': i+1,
        'Environment': result.environment.__class__.__name__,
        'Algorithm': result.policy.__class__.__name__,
        'Policy_Name': result.policy.name,
        'Best_Parameters': str(result.chosen_hyper_parameters),
    })

comparison_df = pd.DataFrame(comparison_data)
print("=== Optimization Results Comparison ===")
print(comparison_df.to_string(index=False))

# Use optimized policies for further analysis
print(f"\nSuccessfully optimized {len(optimized_policies)} policies:")
for policy in optimized_policies:
    print(f"  - {policy.name} ({policy.__class__.__name__})")

# Test one of the optimized policies
if optimized_policies:
    test_policy = optimized_policies[0]
    print(f"\nTesting optimized policy: {test_policy.name}")
    
    # Get action from optimized policy
    test_belief = tiger_belief
    action, run_data = test_policy.action(test_belief)
    
    print(f"Optimized policy action: {action}")
    
    # Extract planning time from info_variables list
    planning_time = 0
    for info_var in run_data.info_variables:
        if info_var.name == 'planning_time':
            planning_time = info_var.value
            break
    
    print(f"Planning time: {planning_time:.3f} seconds")

## Best Practices for Hyperparameter Optimization

Ideally, we want to perform hyper-parameter tuning, and then evaluated the chosen model afterwards. 

In [ ]:
"""
Example using the LocalSimulationsAPI run_optimize_and_evaluate method.

This method runs hyperparameter optimization followed by evaluation of the optimized
policies using local Joblib parallelization.
"""

from pathlib import Path
from POMDPPlanners.simulations.simulation_apis.local_simulations_api import LocalSimulationsAPI
from POMDPPlanners.planners.mcts_planners.pomcpow import POMCPOW
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.core.simulation import NumericalHyperParameter
from POMDPPlanners.core.simulation.hyperparameter_tuning import (
    HyperParameterRunParams,
    HyperParameterOptimizationDirection,
)
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs
from POMDPPlanners.utils.action_samplers import DiscreteActionSampler
import pandas as pd

# Setup environment and initial belief
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
environment, initial_belief = env_configs.tiger_pomdp_config(n_particles=200)

action_sampler = DiscreteActionSampler(actions=environment.get_actions())

print(f"Environment: {environment.name}")
print(f"Initial belief particles: {len(initial_belief.particles)}")

# Configure multiple planners for optimization and evaluation
planner_configs = PlannersHyperparamConfigs(discount_factor=0.95)

# Create hyperparameter run configurations
optimization_configs = [
    # POMCPOW configuration
    HyperParameterRunParams(
        environment=environment,
        belief=initial_belief,
        hyper_param_planner_config=planner_configs.pomcpow_config(
            env=environment,
            action_sampler=action_sampler,
            name="OptimizedPOMCPOW"
        ),
        num_episodes=50,  # Optimization episodes
        num_steps=50,     # Optimization steps
        n_trials=50,      # Number of optimization trials
        parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
    ),

    # SparsePFT configuration
    HyperParameterRunParams(
        environment=environment,
        belief=initial_belief,
        hyper_param_planner_config=planner_configs.sparse_pft_config(
            env=environment,
            name="OptimizedSparsePFT"
        ),
        num_episodes=50,  # Optimization episodes
        num_steps=50,     # Optimization steps
        n_trials=50,      # Number of optimization trials
        parameters_to_optimize=[("average_return", HyperParameterOptimizationDirection.MAXIMIZE)]
    )
]

# Set up cache directory
cache_dir = Path("./api_optimization_results")
cache_dir.mkdir(exist_ok=True)

api = LocalSimulationsAPI(debug=True)

# Run optimization and evaluation
results, stats_df = api.run_optimize_and_evaluate(
    configs=optimization_configs,
    evaluation_episodes=50,      # Evaluation episodes
    evaluation_steps=50,          # Evaluation steps
    evaluation_n_jobs=1,          # Parallel jobs for evaluation
    optimization_n_jobs=-1,       # Parallel jobs for optimization
    confidence_interval_level=0.95,
    alpha=0.05,
    cache_dir_path=cache_dir,
    experiment_name="api_tiger_optimization",
    debug=False,
    cache_visualizations=True
)

# Display evaluation statistics
print("\n=== Evaluation Results ===")
print(stats_df[['environment_name', 'policy_name', 'mean_total_return', 'ci_lower', 'ci_upper']])

### Trial Configuration
- Start with fewer trials (50-100) for initial exploration
- Increase trials (200-500) for final optimization
- Use more episodes (50-100) for reliable performance estimates

### Computational Resources
- Use parallel execution (`n_jobs=-1`) when available
- Consider using distributed computing for large-scale optimization
- Monitor memory usage with large particle counts

### Evaluation Strategy
- Use consistent evaluation metrics across all configurations
- Consider multiple performance metrics (average return, success rate, planning time)
- Validate optimized policies on held-out test episodes

## Viewing Detailed Results with MLflow

The optimization and evaluation process automatically saves detailed results and metrics to MLflow tracking. To explore these results interactively:

### 1. Navigate to the cache directory
```bash
cd ./api_optimization_results
```

### 2. Launch MLflow UI
```bash
mlflow ui
```

### 3. Open browser to view results
Navigate to `http://localhost:5000` in your browser to access the MLflow UI.

### What you can explore in MLflow:

- **Hyperparameter optimization trials and metrics**: View how different parameter combinations performed during optimization
- **Policy evaluation results and statistics**: Detailed performance metrics for each optimized policy
- **Experiment comparison**: Compare results across different runs and experiments
- **Parameter importance**: Understand which hyperparameters have the most impact on performance
- **Optimization convergence plots**: Visual tracking of how optimization progressed over trials
- **Statistical confidence intervals**: Detailed statistical analysis with confidence bounds
- **Performance comparisons**: Side-by-side comparison of different policies

The MLflow UI provides an interactive dashboard that makes it easy to analyze and understand your optimization and evaluation results in depth.

## Troubleshooting Common Issues

**Low Performance After Optimization**
- Check if parameter ranges are appropriate for the problem
- Verify that the optimization direction is correct (maximize vs minimize)
- Ensure sufficient trials and episodes for reliable estimates

**Optimization Taking Too Long**
- Reduce the number of trials or episodes
- Use fewer particles in belief representation
- Decrease the planning depth or timeout limits

**Memory Issues**
- Reduce particle count in belief representation
- Use smaller planning depths
- Consider using sparse belief representations

**Convergence Problems**
- Increase the number of trials
- Adjust parameter ranges based on initial results
- Consider using different optimization algorithms in Optuna

## Summary

This notebook demonstrated:

1. **Basic hyperparameter optimization** for POMCP on Tiger POMDP
2. **Multi-environment optimization** comparing different algorithms
3. **Predefined configurations** for quick setup
4. **Results analysis** and policy comparison
5. **Best practices** for effective optimization

The optimization framework provides powerful tools for finding optimal hyperparameters across different POMDP environments and planning algorithms. For production use, consider:

- Increasing trial counts and episode numbers
- Using multiple CPU cores (`n_jobs=-1`)
- Running longer evaluations for statistical significance
- Comparing results across multiple random seeds

## What's Next?

- **Advanced optimization**: See [advanced_optimization.ipynb](advanced_optimization.ipynb) for multi-objective optimization, risk-averse optimization, and research workflows
- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for batch evaluations with statistical analysis
- **Inspect planner behavior**: See [tree_analysis_debugging.ipynb](tree_analysis_debugging.ipynb) for MCTS tree metrics, visualization, and debugging
- **Basic usage**: See [basic_usage.ipynb](basic_usage.ipynb) for the core POMDP interaction loop